In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df = pd.read_csv('../data/raw/customer_support_tickets.csv')
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head(2)

Loaded 8,469 rows, 17 columns


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN


In [2]:
# Store original names for reference, then rename
print("Before (original names):")
print(df.columns.tolist())

df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

print("\nAfter (cleaned names):")
print(df.columns.tolist())

Before (original names):
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

After (cleaned names):
['ticket_id', 'customer_name', 'customer_email', 'customer_age', 'customer_gender', 'product_purchased', 'date_of_purchase', 'ticket_type', 'ticket_subject', 'ticket_description', 'ticket_status', 'resolution', 'ticket_priority', 'ticket_channel', 'first_response_time', 'time_to_resolution', 'customer_satisfaction_rating']


In [3]:
# Columns that should be proper datetimes
date_cols = ['date_of_purchase', 'first_response_time', 'time_to_resolution']

print("BEFORE parsing:")
print(df[date_cols].dtypes)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("\nAFTER parsing:")
print(df[date_cols].dtypes)

# Sanity check: did we lose any values during parsing?
print("\nNon-null counts after parse:")
for col in date_cols:
    print(f"  {col}: {df[col].notna().sum():,}")

BEFORE parsing:
date_of_purchase       str
first_response_time    str
time_to_resolution     str
dtype: object

AFTER parsing:
date_of_purchase       datetime64[us]
first_response_time    datetime64[us]
time_to_resolution     datetime64[us]
dtype: object

Non-null counts after parse:
  date_of_purchase: 8,469
  first_response_time: 5,650
  time_to_resolution: 2,769


In [4]:
# Categoricals: strip whitespace, title-case for consistent display
cat_cols = ['ticket_type', 'ticket_priority', 'ticket_channel', 'ticket_status', 'customer_gender']

for col in cat_cols:
    before_unique = df[col].nunique()
    df[col] = df[col].astype(str).str.strip().str.title()
    after_unique = df[col].nunique()
    print(f"{col}: {before_unique} unique → {after_unique} unique")

print("\nPriority values after cleaning:")
print(df['ticket_priority'].value_counts())

print("\nStatus values after cleaning:")
print(df['ticket_status'].value_counts())

ticket_type: 5 unique → 5 unique
ticket_priority: 4 unique → 4 unique
ticket_channel: 4 unique → 4 unique
ticket_status: 3 unique → 3 unique
customer_gender: 3 unique → 3 unique

Priority values after cleaning:
ticket_priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

Status values after cleaning:
ticket_status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64


In [6]:
# Calculate raw resolution time in hours
raw_resolution = (
    (df['time_to_resolution'] - df['first_response_time']).dt.total_seconds() / 3600
)

# Data quality flag: mark rows where raw duration was negative
df['dq_negative_duration'] = raw_resolution < 0

# Use absolute value — treats negative durations as a timestamp-ordering bug
df['resolution_time_hours'] = raw_resolution.abs().round(2)

# Summary
print("Resolution time after fix:")
print(df['resolution_time_hours'].describe())

print(f"\nRows with negative duration flag: {df['dq_negative_duration'].sum():,}")
print(f"Rows with clean duration: {(~df['dq_negative_duration'] & df['resolution_time_hours'].notna()).sum():,}")
print(f"Rows without resolution (Open/Pending): {df['resolution_time_hours'].isna().sum():,}")

Resolution time after fix:
count    2769.000000
mean        7.742412
std         5.613373
min         0.000000
25%         3.020000
50%         6.700000
75%        11.650000
max        23.470000
Name: resolution_time_hours, dtype: float64

Rows with negative duration flag: 1,365
Rows with clean duration: 1,404
Rows without resolution (Open/Pending): 5,700


In [7]:
# Industry-standard SLA thresholds by priority
sla_targets = {
    'Critical': 4,
    'High': 24,
    'Medium': 48,
    'Low': 72
}

df['sla_target_hours'] = df['ticket_priority'].map(sla_targets)

# Verify every ticket got mapped (no unmapped priorities)
print("SLA target distribution:")
print(df['sla_target_hours'].value_counts(dropna=False).sort_index())

print(f"\nUnmapped rows (should be 0): {df['sla_target_hours'].isna().sum()}")

SLA target distribution:
sla_target_hours
4     2129
24    2085
48    2192
72    2063
Name: count, dtype: int64

Unmapped rows (should be 0): 0


In [8]:
# SLA breach flag: True if resolution took longer than target
# Only meaningful for Closed tickets (others have no resolution_time_hours)
df['sla_breached'] = np.where(
    df['resolution_time_hours'].notna() &
    (df['resolution_time_hours'] > df['sla_target_hours']),
    True,
    False
)

# Summary by priority
print("SLA breach rate by priority:")
breach_summary = df[df['resolution_time_hours'].notna()].groupby('ticket_priority').agg(
    total_closed=('ticket_id', 'count'),
    breached=('sla_breached', 'sum'),
    breach_rate=('sla_breached', 'mean'),
    avg_resolution_hours=('resolution_time_hours', 'mean')
).round(3)
print(breach_summary)

print(f"\nOverall breach rate (Closed tickets): {df[df['resolution_time_hours'].notna()]['sla_breached'].mean()*100:.1f}%")

SLA breach rate by priority:
                 total_closed  breached  breach_rate  avg_resolution_hours
ticket_priority                                                           
Critical                  726       481        0.663                 7.564
High                      705         0        0.000                 8.201
Low                       644         0        0.000                 7.858
Medium                    694         0        0.000                 7.355

Overall breach rate (Closed tickets): 17.4%


In [9]:
# Aging buckets: categorize resolution times into business-friendly ranges
def age_bucket(hours):
    if pd.isna(hours):
        return 'Unresolved'
    if hours <= 24:
        return '0-24h'
    if hours <= 48:
        return '24-48h'
    if hours <= 72:
        return '48-72h'
    return '72h+'

df['aging_bucket'] = df['resolution_time_hours'].apply(age_bucket)

# Verify
print("Aging bucket distribution:")
print(df['aging_bucket'].value_counts(dropna=False))

print(f"\nTotal rows: {len(df):,}")

Aging bucket distribution:
aging_bucket
Unresolved    5700
0-24h         2769
Name: count, dtype: int64

Total rows: 8,469


In [10]:
# Time dimensions built from first_response_time (the ticket-start proxy)
# Used for month-over-month trends, weekly patterns, day-of-week analysis

df['response_year'] = df['first_response_time'].dt.year
df['response_month'] = df['first_response_time'].dt.to_period('M').astype(str)
df['response_month_name'] = df['first_response_time'].dt.strftime('%b %Y')
df['response_week'] = df['first_response_time'].dt.isocalendar().week
df['response_day_of_week'] = df['first_response_time'].dt.day_name()
df['response_date'] = df['first_response_time'].dt.date

# Verify
print("Time dimension samples:")
print(df[['first_response_time', 'response_year', 'response_month', 
         'response_month_name', 'response_week', 'response_day_of_week']].head(5))

print("\nMonth distribution:")
print(df['response_month'].value_counts(dropna=False).sort_index())

print("\nDay-of-week distribution:")
print(df['response_day_of_week'].value_counts(dropna=False))

Time dimension samples:
  first_response_time  response_year response_month response_month_name  response_week response_day_of_week
0 2023-06-01 12:15:36         2023.0        2023-06            Jun 2023             22             Thursday
1 2023-06-01 16:45:38         2023.0        2023-06            Jun 2023             22             Thursday
2 2023-06-01 11:14:38         2023.0        2023-06            Jun 2023             22             Thursday
3 2023-06-01 07:29:40         2023.0        2023-06            Jun 2023             22             Thursday
4 2023-06-01 00:12:42         2023.0        2023-06            Jun 2023             22             Thursday

Month distribution:
response_month
2023-05     175
2023-06    5475
NaN        2819
Name: count, dtype: int64

Day-of-week distribution:
response_day_of_week
Thursday     5437
NaN          2819
Wednesday     175
Friday         38
Name: count, dtype: int64


In [11]:
# Hour-of-day dimension — useful since timestamps cluster in a ~48hr window
df['response_hour'] = df['first_response_time'].dt.hour

# Quick sanity check
print("Tickets by hour of day:")
print(df['response_hour'].value_counts(dropna=False).sort_index())

Tickets by hour of day:
response_hour
0.0      240
1.0      239
2.0      253
3.0      206
4.0      243
5.0      247
6.0      236
7.0      242
8.0      225
9.0      221
10.0     219
11.0     260
12.0     230
13.0     199
14.0     234
15.0     253
16.0     207
17.0     234
18.0     233
19.0     292
20.0     223
21.0     244
22.0     249
23.0     221
NaN     2819
Name: count, dtype: int64


In [12]:
# Columns to export to Power BI — keep analytical columns, drop PII and noise
cols_to_export = [
    'ticket_id',
    'customer_age', 'customer_gender',
    'product_purchased', 'date_of_purchase',
    'ticket_type', 'ticket_subject',
    'ticket_status', 'ticket_priority', 'ticket_channel',
    'first_response_time', 'time_to_resolution',
    'resolution_time_hours', 'sla_target_hours', 'sla_breached',
    'aging_bucket', 'dq_negative_duration',
    'customer_satisfaction_rating',
    'response_year', 'response_month', 'response_month_name',
    'response_week', 'response_day_of_week', 'response_date', 'response_hour'
]

df_clean = df[cols_to_export].copy()

# Export
output_path = '../data/processed/tickets_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Exported {len(df_clean):,} rows × {df_clean.shape[1]} columns")
print(f"File: {output_path}")
print(f"\nColumn list:")
for c in df_clean.columns:
    print(f"  - {c}")

Exported 8,469 rows × 25 columns
File: ../data/processed/tickets_clean.csv

Column list:
  - ticket_id
  - customer_age
  - customer_gender
  - product_purchased
  - date_of_purchase
  - ticket_type
  - ticket_subject
  - ticket_status
  - ticket_priority
  - ticket_channel
  - first_response_time
  - time_to_resolution
  - resolution_time_hours
  - sla_target_hours
  - sla_breached
  - aging_bucket
  - dq_negative_duration
  - customer_satisfaction_rating
  - response_year
  - response_month
  - response_month_name
  - response_week
  - response_day_of_week
  - response_date
  - response_hour


In [13]:
# Date dimension: one row per date in the data range
from datetime import timedelta

min_date = df['first_response_time'].min().normalize()
max_date = df['first_response_time'].max().normalize()

# Pad with extra days so slicers feel natural
date_range = pd.date_range(min_date - timedelta(days=2), 
                           max_date + timedelta(days=2), 
                           freq='D')

date_dim = pd.DataFrame({'date': date_range})
date_dim['year']        = date_dim['date'].dt.year
date_dim['quarter']     = date_dim['date'].dt.quarter
date_dim['month']       = date_dim['date'].dt.month
date_dim['month_name']  = date_dim['date'].dt.strftime('%b %Y')
date_dim['week']        = date_dim['date'].dt.isocalendar().week
date_dim['day']         = date_dim['date'].dt.day
date_dim['day_of_week'] = date_dim['date'].dt.day_name()
date_dim['is_weekend']  = date_dim['date'].dt.dayofweek.isin([5, 6])

date_dim.to_csv('../data/processed/date_dim.csv', index=False)

print(f"Date dimension: {len(date_dim)} rows")
print(f"Range: {date_dim['date'].min().date()} to {date_dim['date'].max().date()}")
print(date_dim.head(10))

Date dimension: 7 rows
Range: 2023-05-29 to 2023-06-04
        date  year  quarter  month month_name  week  day day_of_week  is_weekend
0 2023-05-29  2023        2      5   May 2023    22   29      Monday       False
1 2023-05-30  2023        2      5   May 2023    22   30     Tuesday       False
2 2023-05-31  2023        2      5   May 2023    22   31   Wednesday       False
3 2023-06-01  2023        2      6   Jun 2023    22    1    Thursday       False
4 2023-06-02  2023        2      6   Jun 2023    22    2      Friday       False
5 2023-06-03  2023        2      6   Jun 2023    22    3    Saturday        True
6 2023-06-04  2023        2      6   Jun 2023    22    4      Sunday        True
